In [ ]:
import os
import chromadb
import gradio as gr
import anthropic
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

load_dotenv()

print("Loading models...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="cricket_docs")
claude = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
print(f"✅ Ready — {collection.count()} chunks loaded\n")

def retrieve_chunks(question, top_k=5):
    question_embedding = embedding_model.encode(question).tolist()
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=top_k
    )
    chunks = results['documents'][0]
    sources = [m['source'] for m in results['metadatas'][0]]
    return chunks, sources

def cricket_chat(question, history):
    if not question.strip():
        return "", history, ""

    chunks, sources = retrieve_chunks(question)
    context = "\n\n".join([f"[Source: {src}]\n{chunk}"
                           for src, chunk in zip(sources, chunks)])

    prompt = f"""You are a cricket expert assistant. Do NOT mention the knowledge base, context, or web search in 
your answer. Just give the answer itself. Just answer the question asked.

KNOWLEDGE BASE CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

    response = claude.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1024,
        tools=[
            {
                "type": "web_search_20250305",
                "name": "web_search"
            }
        ],
        messages=[{"role": "user", "content": prompt}]
    )

    answer = ""
    for block in response.content:
        if block.type == "text":
            answer += block.text

    unique_sources = list(set(sources))
    sources_text = "**Sources used:**\n" + "\n".join([f"- {s}" for s in unique_sources])

    history.append({"role": "user", "content": question})
    history.append({"role": "assistant", "content": answer})

    return "", history, sources_text

with gr.Blocks(theme=gr.themes.Soft(), title="🏏 Cricket RAG Chatbot") as demo:

    gr.Markdown("""
    # 🏏 Cricket RAG Chatbot
    Ask me anything about cricket — players, rules, tournaments, records and more!
    """)

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=450, label="Chat")
            question_box = gr.Textbox(
                placeholder="e.g. Who won the IPL 2025?",
                label="Your Question",
                lines=2
            )
            with gr.Row():
                submit_btn = gr.Button("Ask 🏏", variant="primary")
                clear_btn = gr.Button("Clear")

        with gr.Column(scale=1):
            sources_box = gr.Markdown(label="Sources", value="Sources will appear here...")

    gr.Examples(
        examples=[
            "How many international centuries did Sachin Tendulkar score?",
            "What is LBW in cricket?",
            "Who won the IPL 2025?",
            "What is the difference between Test and ODI cricket?",
            "Who is the current number 1 ranked Test batsman?"
        ],
        inputs=question_box
    )

    submit_btn.click(
        fn=cricket_chat,
        inputs=[question_box, chatbot],
        outputs=[question_box, chatbot, sources_box]
    )
    question_box.submit(
        fn=cricket_chat,
        inputs=[question_box, chatbot],
        outputs=[question_box, chatbot, sources_box]
    )
    clear_btn.click(lambda: ([], ""), outputs=[chatbot, sources_box])

demo.launch()

Loading models...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4518.31it/s]


✅ Ready — 496 chunks loaded



/var/folders/bc/_s4bf2c52m595yz7wrdw0pyc0000gn/T/ipykernel_72350/3020859354.py:71: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="🏏 Cricket RAG Chatbot") as demo:


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
